# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [13]:
from google.colab import userdata
userdata.get('GEMINI_API_KEY')

'AQ.Ab8RN6KpwWAEqVpX73WXNRwhP15bw0B6MpLhv5FpzhsbGkzKQg'

In [14]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [15]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-datastore 2.27.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-bigquery-storage 2.43.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-functions 1.25.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
grpc-google-iam-v1 0.14.6 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5

In [16]:

# @title User input needed

DOCUMENTS = {
    "ai_basics": (
        "Artificial intelligence is a branch of computer science "
        "that develops systems capable of performing tasks that "
        "normally require human intelligence. These tasks include "
        "learning, reasoning, problem-solving, and understanding language. "
        "AI systems process data to identify patterns and make predictions. "
        "Examples include virtual assistants, recommendation systems, "
        "and chatbots."
    ),

    "machine_learning": (
        "Machine learning is a subset of artificial intelligence "
        "that enables computers to learn patterns from data without "
        "being explicitly programmed for every task. Supervised learning "
        "uses labelled data. Unsupervised learning discovers patterns "
        "without predefined labels. Deep learning uses neural networks "
        "with multiple layers to learn complex patterns."
    )
}

print("Documents loaded successfully:", list(DOCUMENTS.keys()))


Documents loaded successfully: ['ai_basics', 'machine_learning']


In [17]:

# @title Define a tool

def search_documents(query: str) -> str:
    """Search the document collection for relevant paragraphs."""

    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())

        match_count = sum(
            1 for word in words
            if word in content.lower()
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []

    for _, name, content in results[:2]:
        output.append(f"Source: {name}\n{content}")

    return "\n\n".join(output)


# Sanity check
print(search_documents("minimum of 4"))


Source: machine_learning
Machine learning is a subset of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed for every task. Supervised learning uses labelled data. Unsupervised learning discovers patterns without predefined labels. Deep learning uses neural networks with multiple layers to learn complex patterns.

Source: ai_basics
Artificial intelligence is a branch of computer science that develops systems capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, and understanding language. AI systems process data to identify patterns and make predictions. Examples include virtual assistants, recommendation systems, and chatbots.


In [18]:
!pip install -q google-generativeai

In [19]:
import google.generativeai as genai

print("Gemini library installed successfully!")


Gemini library installed successfully!


/tmp/ipykernel_9221/2744270880.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [20]:


# @title Ask Gemini to search for something

import google.generativeai as genai
import time
import os
from google.colab import userdata

# Load Gemini API key from Colab Secrets
genai.configure(
    api_key=userdata.get("GEMINI_API_KEY")
)

# Initialize Gemini model
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    tools=[search_documents],
    system_instruction=(
        """
        You are an AI assistant answering questions about Artificial Intelligence.
        Use the search_documents tool when you need information from the documents.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer, say so clearly.
        """
    )
)

print("Model ready:", model.model_name)


Model ready: models/gemini-2.5-flash


In [21]:

# @title Define documents and search tool

# Step 1: Define the document collection
DOCUMENTS = {
    "ai_basics": (
        "Artificial intelligence is a branch of computer science "
        "that develops systems capable of performing tasks that "
        "normally require human intelligence. These tasks include "
        "learning, reasoning, problem-solving, and understanding "
        "language. AI systems process data to identify patterns "
        "and make predictions. Examples include virtual assistants, "
        "recommendation systems, and chatbots."
    ),

    "machine_learning": (
        "Machine learning is a subset of artificial intelligence "
        "that enables computers to learn patterns from data without "
        "being explicitly programmed for every task. Supervised "
        "learning uses labelled data. Unsupervised learning "
        "discovers patterns without predefined labels. Deep learning "
        "uses neural networks with multiple layers to learn complex patterns."
    )
}


# Step 2: Define the search function
def search_documents(query: str) -> str:

    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())

        match_count = sum(
            1 for word in words
            if word in content.lower()
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    output = []

    for _, name, content in results[:2]:
        output.append(f"Source: {name}\n{content}")

    return "\n\n".join(output)


# Step 3: Test the search function
print("Documents loaded:", list(DOCUMENTS.keys()))
print("\nSearch test:")
print(search_documents("artificial intelligence"))



Documents loaded: ['ai_basics', 'machine_learning']

Search test:
Source: machine_learning
Machine learning is a subset of artificial intelligence that enables computers to learn patterns from data without being explicitly programmed for every task. Supervised learning uses labelled data. Unsupervised learning discovers patterns without predefined labels. Deep learning uses neural networks with multiple layers to learn complex patterns.

Source: ai_basics
Artificial intelligence is a branch of computer science that develops systems capable of performing tasks that normally require human intelligence. These tasks include learning, reasoning, problem-solving, and understanding language. AI systems process data to identify patterns and make predictions. Examples include virtual assistants, recommendation systems, and chatbots.


In [22]:
!pip install -q google-genai

In [23]:

# @title Test Gemini connection

from google import genai
from google.colab import userdata

print("Testing Gemini...", flush=True)

try:
    client = genai.Client(
        api_key=userdata.get("GEMINI_API_KEY")
    )

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents="Reply with only the word Hello."
    )

    print("Success:", response.text)

except Exception as e:
    print("Error:", type(e).__name__, str(e))


Testing Gemini...
Success: Hello


In [24]:

# @title Initialize Gemini Client

from google import genai
from google.colab import userdata

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

print("Gemini client initialized successfully!")


Gemini client initialized successfully!


In [25]:

# @title Define the AI Agent

from google.genai import types

def run_agent(question):

    print("Question:", question)

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=question,
        config=types.GenerateContentConfig(
            tools=[search_documents],
            system_instruction=(
                "You are an AI assistant answering questions "
                "about Artificial Intelligence. "
                "Use the search_documents tool when needed. "
                "Only answer based on the documents. "
                "If the documents do not contain the answer, "
                "say so clearly."
            ),
            max_output_tokens=500
        )
    )

    print("Answer:", response.text)

print("AI Agent function defined successfully!")


AI Agent function defined successfully!


In [26]:

# @title Ask follow-up questions

import time

questions = [
    "What is the difference between artificial intelligence and machine learning?",
    "How many ghosts eat carrots every day?",
    "What are the key ethical concerns when developing AI systems?",
    "What are the main applications of artificial intelligence in education?",
    "Why are fairness, privacy, and human oversight important in AI ethics?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)

    print("\n" + "=" * 60)
    run_agent(q)




Question: What is the difference between artificial intelligence and machine learning?
Answer: Based on the provided documents, the difference is that **machine learning is a subset of artificial intelligence**. 

* **Artificial Intelligence (AI)** is a broader branch of computer science that develops systems capable of performing tasks requiring human intelligence, such as learning, reasoning, problem-solving, and understanding language.
* **Machine Learning (ML)** is a specific subset of AI that focuses on enabling computers to learn patterns from data without being explicitly programmed for every task.

Question: How many ghosts eat carrots every day?
Answer: The provided documents do not contain any information about ghosts eating carrots.

Question: What are the key ethical concerns when developing AI systems?
Answer: Based on the provided documents, there is no mention of the ethical concerns when developing AI systems. Therefore, I cannot answer this question using the availabl

In [27]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)